# Notebook 4: Lick Decoder
Logistic regression on PCA-reduced population activity (session 1044385384).

**Before running:** add your GitHub token as a Colab Secret:
1. Click the 🔑 key icon in the left sidebar
2. Add a secret named `GITHUB_TOKEN` with your Personal Access Token as the value
3. Toggle it on for this notebook

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os, sys

REPO = "/content/allen-neuropixels-decoder"

# Clone repo if not already present; pull latest if it is
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/jadenbalajadia/allen-neuropixels-decoder.git {REPO}")
else:
    os.system(f"git -C {REPO} pull")

sys.path.append(REPO)

from google.colab import drive, userdata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/lickingpcaoutputs")
print("Setup complete. DATA_DIR:", DATA_DIR)

In [ ]:
# ── Cell 2: Load data ──────────────────────────────────────────────────────
from src.preprocessing import load_arrays

Z, X, y, labels = load_arrays(DATA_DIR)
print(f"Z: {Z.shape}  X: {X.shape}  y: {y.shape}")
print(f"Class balance — lick bins: {y.mean()*100:.1f}%  no-lick bins: {(1-y.mean())*100:.1f}%")

In [ ]:
# ── Cell 3: Cross-validation ───────────────────────────────────────────────
# Note: raw accuracy is misleading here (~5% lick bins means chance = 94.8%).
# Balanced accuracy and F1 give a fair picture; AUC is the primary metric.
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.decoder import build_decoder

clf = build_decoder()
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bal_acc = cross_val_score(clf, Z, y, cv=cv, scoring="balanced_accuracy")
f1      = cross_val_score(clf, Z, y, cv=cv, scoring="f1")

print(f"Balanced accuracy : {bal_acc.mean():.3f} ± {bal_acc.std():.3f}")
print(f"F1 score          : {f1.mean():.3f} ± {f1.std():.3f}")

In [ ]:
# ── Cell 4: ROC curve ──────────────────────────────────────────────────────
from src.decoder import plot_roc

os.makedirs(f"{REPO}/figures", exist_ok=True)

fig, mean_auc = plot_roc(Z, y, n_splits=5)
fig.savefig(f"{REPO}/figures/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean AUC: {mean_auc:.3f}")

In [ ]:
# ── Cell 5: Push figure to GitHub ──────────────────────────────────────────
# Reads your token from Colab Secrets — never hardcode it in the notebook.
import subprocess

token = userdata.get("GITHUB_TOKEN")  # set this in the 🔑 Secrets panel

def git(cmd):
    result = subprocess.run(
        ["git", "-C", REPO] + cmd,
        capture_output=True, text=True
    )
    if result.stdout: print(result.stdout.strip())
    if result.stderr: print(result.stderr.strip())

git(["config", "user.email", "jadenbalajadia4@gmail.com"])
git(["config", "user.name",  "jadenbalajadia"])
git(["add",    "figures/roc_curve.png"])
git(["commit", "-m", "Add ROC curve figure (AUC 0.921)"])
git(["push",   f"https://{token}@github.com/jadenbalajadia/allen-neuropixels-decoder.git"])

print("Done — check github.com/jadenbalajadia/allen-neuropixels-decoder")